# Viz 2 — Machine Learning Charts

Renders the ML-tier charts: regularised-coefficient comparison, Random
Forest feature importance, and the 2020–2030 forecast for top-3 improvers
and bottom-3 decliners.

| Chart | Page heading | Artifact |
|---|---|---|
| 7 | Coefficient Agreement: LASSO, Ridge, Elastic Net | ml_coefficients.csv |
| 8 | Random Forest Feature Importance | ml_coefficients.csv |
| 9 | Projected ECI Trajectories (2020–2030) | ml_forecast.csv + ml_forecast_history.csv |

All ML fitting happens in `prep/3_ml_models.ipynb`. This notebook only
renders.


In [1]:
import os, sys
sys.path.insert(0, '.')
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from _style import (base_layout, save, hex_rgba, artifact_path,
                    NAVY, GRID, BG)

coef = pd.read_csv(artifact_path('ml_coefficients.csv'))

# Drop the lagged ECI from coefficient charts (its near-unit value would
# dwarf every other regressor visually).
coef_show = coef[coef['short'] != 'Lagged ECI'].copy()


## Chart 7 — Coefficient agreement (top 12)

Top 12 features by mean absolute coefficient across the three regularised
models. Three bars per feature so the reader can see whether the models
agree on sign and magnitude.


In [2]:
top12 = coef_show.sort_values('abs_avg', ascending=True).tail(12).copy()

MC = {'LASSO': '#c23a3a', 'Ridge': '#3498DB', 'Elastic Net': '#2e7d4a'}
fig07 = go.Figure()
fig07.add_vline(x=0, line=dict(color='#c9cfd6', width=1.5))
for m, col in MC.items():
    key = {'LASSO': 'lasso', 'Ridge': 'ridge', 'Elastic Net': 'en'}[m]
    fig07.add_trace(go.Bar(
        y=top12['short'], x=top12[key], name=m, orientation='h',
        marker=dict(color=col, opacity=0.88,
                    line=dict(color='white', width=0.5)),
        hovertemplate=f'%{{y}}: %{{x:+.4f}}<extra>{m}</extra>',
    ))
fig07.update_layout(
    **base_layout(height=560, margin=dict(l=165, r=40, t=20, b=70)),
    barmode='group',
    xaxis=dict(title='Coefficient (standardised inputs)', gridcolor=GRID),
    yaxis=dict(gridcolor=GRID, tickfont=dict(size=11)),
    legend=dict(x=1.01, y=0.99, font=dict(size=11),
                bgcolor='rgba(250,250,250,0.9)',
                bordercolor=GRID, borderwidth=1),
)
save(fig07, '07_lasso_ridge_en_coefficients.html')


  wrote outputs/07_lasso_ridge_en_coefficients.html


## Chart 8 — Random Forest feature importance (top 12)

MDI importance, normalised to 0–1 inside the prep notebook. Top 12 by
that normalised score.


In [3]:
rf12 = coef_show.sort_values('rf_norm', ascending=True).tail(12).copy()

fig08 = go.Figure(go.Bar(
    y=rf12['short'], x=rf12['rf_norm'], orientation='h',
    marker=dict(color='#c97030', opacity=0.9,
                line=dict(color='white', width=0.5)),
    hovertemplate='%{y}: %{x:.4f}<extra>Random Forest (MDI)</extra>',
))
fig08.update_layout(
    **base_layout(height=500, margin=dict(l=165, r=40, t=20, b=70)),
    xaxis=dict(title='Feature Importance (MDI, normalised)', gridcolor=GRID),
    yaxis=dict(gridcolor=GRID, tickfont=dict(size=11)),
    showlegend=False,
)
save(fig08, '08_rf_feature_importance.html')


  wrote outputs/08_rf_feature_importance.html


## Chart 9 — ECI forecast (top-3 improvers, bottom-3 decliners)

Two-panel plot. Left: top three improvers, ranked by total ensemble change
2019 → 2030. Right: bottom three. Solid lines are historical, dashed are
the ensemble forecast, the band is ±1 SD across the four models.

Background lines for non-highlighted countries are intentionally not drawn
(reduces visual noise — the page heading is already explicit about scope).


In [4]:
forecast = pd.read_csv(artifact_path('ml_forecast.csv'))
hist     = pd.read_csv(artifact_path('ml_forecast_history.csv'))

ranking = (forecast.groupby('Country Code')
           .agg(Country=('Country Name', 'first'),
                ECI_2019=('Last_Known_ECI', 'first'),
                ECI_2030=('Ensemble', 'last'))
           .reset_index())
ranking['Total_Change'] = ranking['ECI_2030'] - ranking['ECI_2019']
ranking = ranking.sort_values('Total_Change', ascending=False).reset_index(drop=True)

top3    = ranking.head(3)['Country Code'].tolist()
bottom3 = ranking.tail(3)['Country Code'].tolist()
cc_to_name = dict(zip(ranking['Country Code'], ranking['Country']))
print(f'Top 3 improvers:    {top3}')
print(f'Bottom 3 decliners: {bottom3}')

TOP_C = '#2e7d4a'
BOT_C = '#c23a3a'

figZ = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.08, shared_yaxes=True,
    subplot_titles=[
        f"Top 3 improvers: {' · '.join(cc_to_name.get(c, c) for c in top3)}",
        f"Bottom 3 decliners: {' · '.join(cc_to_name.get(c, c) for c in bottom3)}",
    ],
)

for panel_i, (highlight, h_col) in enumerate([(top3, TOP_C), (bottom3, BOT_C)], 1):
    for cc in highlight:
        h = hist[hist['Country Code'] == cc].sort_values('Year')
        f = forecast[forecast['Country Code'] == cc].sort_values('Year')
        if len(h) == 0:
            continue
        cname = h['Country Name'].iloc[0]
        figZ.add_trace(go.Scatter(
            x=h['Year'], y=h['Economic Complexity Index'], mode='lines',
            line=dict(color=h_col, width=2.2), showlegend=False,
            name=cname,
            hovertemplate=f'<b>{cname}</b><br>%{{x}}: ECI=%{{y:.3f}}<extra></extra>',
        ), row=1, col=panel_i)
        if len(f) == 0:
            continue
        last_y = float(h['Economic Complexity Index'].iloc[-1])
        last_x = int(h['Year'].iloc[-1])
        bx   = [last_x] + f['Year'].tolist()
        by   = [last_y] + f['Ensemble'].tolist()
        bstd = [0.0]    + f['ECI_std'].tolist()
        upper = [y + s for y, s in zip(by, bstd)]
        lower = [y - s for y, s in zip(by, bstd)]
        figZ.add_trace(go.Scatter(
            x=bx + bx[::-1], y=upper + lower[::-1],
            fill='toself', fillcolor=hex_rgba(h_col, 0.10),
            line=dict(width=0), showlegend=False, hoverinfo='skip',
        ), row=1, col=panel_i)
        figZ.add_trace(go.Scatter(
            x=bx, y=by, mode='lines',
            line=dict(color=h_col, width=1.8, dash='dash'),
            showlegend=False, hoverinfo='skip',
        ), row=1, col=panel_i)
        xref = 'x' if panel_i == 1 else 'x2'
        yref = 'y' if panel_i == 1 else 'y2'
        figZ.add_annotation(
            x=2030, y=by[-1], xref=xref, yref=yref,
            text=f'<b>{cc_to_name.get(cc, cc)}</b>',
            showarrow=True, ax=14, ay=0,
            arrowwidth=1, arrowcolor=h_col,
            font=dict(size=9, color=h_col), xanchor='left',
        )
    figZ.add_vline(x=2019.5, line=dict(color='#aaa', width=1.2, dash='dot'),
                   row=1, col=panel_i)
    figZ.update_xaxes(title_text='Year', gridcolor=GRID, dtick=5,
                      range=[1994, 2033], row=1, col=panel_i)

figZ.update_yaxes(title_text='Economic Complexity Index', gridcolor=GRID, row=1, col=1)
figZ.update_yaxes(gridcolor=GRID, row=1, col=2)
figZ.update_layout(
    template='plotly_white', plot_bgcolor=BG, paper_bgcolor=BG,
    font=dict(family='Public Sans, system-ui, sans-serif', size=11, color=NAVY),
    height=580, margin=dict(l=70, r=50, t=40, b=50),
    showlegend=False,
)
save(figZ, '09_eci_forecast_trajectories.html')


Top 3 improvers:    ['MNG', 'GNQ', 'ECU']
Bottom 3 decliners: ['SVN', 'LBY', 'CZE']


  wrote outputs/09_eci_forecast_trajectories.html


## Done

ML charts written. To regenerate them after retraining the models, re-run
`prep/3_ml_models.ipynb` first.
